# Idiom-Aware Fine-Tuning of EN-DE Translations

Idea: 2-stage fine-tuning of a pre-trained model to increase idiom translation accuracy without sacrificing general language translation quality

Specifically in this notebook, I want to investigate whether catastrophic forgetting after idiom-specialized fine-tuning (stage 1) can be mitigated by:
- Dataset mixing (90% WMT + 10% idioms)
- Encoder freezing
- Partial decoder freezing (top 2 and top 4 layers)
- Weight decay
- Early stopping

---
**Stages**

---


**Stage 0**: Baseline (Helsikin-NLP model)

**Stage 1**: Immediate Idiom-only fine-tuning
- Increase the model's idiom translation performance by fine-tuning the baseline from stage 0 on EN-DE idiom sentence pairs (`IdiomsInCtx-MT`)
- This will likely increase idiom-specific translation performance (measured as `Idiom BLEU`), and likely decrease general language translation (measured as `WMT BLEU)` performance.

**Stage 2**: General Language fine-tuning
- Fine-tune the model from stage 1 on a general language EN-DE dataset (`WMT-14`)
- The hope is to maintain the learned idiom-translation ability from stage 1, while recovering some of the lost general language translation perfromance.

Ideally, our stage 2 model has not sacrificied its general language translation capabilties, but has notably increased idiom-specific translation capabilities.

---
**Experiments** in this notebook


---

1. **Mixed Stage 2 + Heavy Regularization**

log_name: *two_stage_mixed_wd_es_v1*

- 90% WMT + 10% idioms
- freeze_encoder=True
- decoder_top_layers_trainable=2
- lr=5e-6
- weight_decay=0.01
- early_stopping=2


2. **Relaxed Mixed Training**

log_name: *two_stage_mixed_wd_es_v2*

- 90% WMT + 10% idioms
- freeze_encoder=True
- decoder_top_layers_trainable=4
- lr=2e-6
- weight_decay=0.00
- early_stopping=2


3. **WMT-only training with extreme freezing**

log_name: *two_stage_wmt_only_freeze_dec1_lr1e5_v1*

- 100% WMT
-	freeze_encoder=True
- decoder_top_layers_trainable=1
- lr=1e-5
- no weight decay
- early stopping

## Clone Repo from Git / Commit Changes
Run the git clone everytime a new session is started

In [ ]:
!rm -rf Idiom-Aware-Finetuning-in-MT-EN-to-DE
!git clone https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
%cd Idiom-Aware-Finetuning-in-MT-EN-to-DE
!ls

Cloning into 'Idiom-Aware-Finetuning-in-MT-EN-to-DE'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 82 (delta 33), reused 30 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 188.04 KiB | 20.89 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/Idiom-Aware-Finetuning-in-MT-EN-to-DE


Copy notebook from Drive into Repo

In [ ]:
# !cp "/content/drive/MyDrive/ds266_idiom_mt/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb" notebooks/

Add and commit changes

In [ ]:
#!git config --global user.email "auntiepersilla@gmail.com"
#!git config --global user.name "Michael Strommer"

#!git add notebooks/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb
#!git commit -m "Add all-in-one Drive-first pipeline notebook"

[main 45cc75b] Add all-in-one Drive-first pipeline notebook
 1 file changed, 1 insertion(+)
 create mode 100644 notebooks/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb


Push to GitHub

In [ ]:
#!git remote set-url origin https://auntiepersilla:<TOKEN>@github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
#!git push origin main

Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 12 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 33.19 KiB | 5.53 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
   1e3ab78..45cc75b  main -> main


## Mount Drive, Create Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
CACHE_DIR = os.path.join(DRIVE_ROOT, "cache")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR

print("DRIVE_ROOT:", DRIVE_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/ds266_idiom_mt
CHECKPOINT_DIR: /content/drive/MyDrive/ds266_idiom_mt/checkpoints
RESULTS_DIR: /content/drive/MyDrive/ds266_idiom_mt/results
CACHE_DIR: /content/drive/MyDrive/ds266_idiom_mt/cache


## Install Dependencies & Imports

In [ ]:
!pip -q install -U "transformers>=4.38" datasets evaluate sacrebleu accelerate sentencepiece sacremoses "protobuf<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 161.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 24.7 MB/s eta 0:00:00


In [ ]:
import random, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
import sacrebleu
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from datasets import concatenate_datasets

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## Load Data

- Idiom-only Data: davidstap/IdiomsInCtx-MT (1000 EN-DE idiom sentence pairs)
  - train: 800
  - dev: 100
  - test: 100
- General Translation Training and Test Data: WMT-14 EN-DE
  - train: 20.000
  - test: 2.000
  - test: 3.003

In [ ]:
def load_idioms(name="davidstap/IdiomsInCtx-MT", config="en-de", train_frac=0.8, dev_frac=0.1, seed=42):
    raw = load_dataset(name, config)

    full = raw["test"] if ("test" in raw and len(raw.keys()) == 1) else raw[list(raw.keys())[0]]
    src_lang, tgt_lang = config.split("-")

    def standardize(ex):
        if "translation" in ex and src_lang in ex["translation"] and tgt_lang in ex["translation"]:
            return {"src": ex["translation"][src_lang], "tgt": ex["translation"][tgt_lang]}
        if src_lang in ex and tgt_lang in ex:
            return {"src": ex[src_lang], "tgt": ex[tgt_lang]}
        raise ValueError(list(ex.keys()))

    full = full.map(standardize)
    full = full.remove_columns([c for c in full.column_names if c not in ["src","tgt"]])

    tmp = full.train_test_split(test_size=(1-train_frac), seed=seed)
    train, rest = tmp["train"], tmp["test"]

    dev_frac_of_rest = dev_frac / (1-train_frac)
    tmp2 = rest.train_test_split(test_size=(1-dev_frac_of_rest), seed=seed)
    dev, test = tmp2["train"], tmp2["test"]

    return DatasetDict({"train": train, "dev": dev, "test": test})

def load_wmt14(ft_train_n=20000, ft_dev_n=2000, seed=42):
    wmt = load_dataset("wmt14", "de-en")
    def to_en_de(ex):
        tr = ex["translation"]
        return {"src": tr["en"], "tgt": tr["de"]}

    train = wmt["train"].map(to_en_de, remove_columns=wmt["train"].column_names)
    test = wmt["test"].map(to_en_de, remove_columns=wmt["test"].column_names)

    shuf = train.shuffle(seed=seed)
    ft_train = shuf.select(range(min(ft_train_n, len(shuf))))
    ft_dev = shuf.select(range(min(ft_train_n, len(shuf)), min(ft_train_n+ft_dev_n, len(shuf))))
    return DatasetDict({"ft_train": ft_train, "ft_dev": ft_dev, "wmt_test": test})

idiom_ds = load_idioms(seed=SEED)
general_ds = load_wmt14(seed=SEED)

print("idiom_ds:", {k: len(v) for k,v in idiom_ds.items()})
print("general_ds:", {k: len(v) for k,v in general_ds.items()})
print("idiom sample:", idiom_ds["test"][0])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Map:   0%|          | 0/3003 [00:00<?, ? examples/s]

idiom_ds: {'train': 800, 'dev': 100, 'test': 100}
general_ds: {'ft_train': 20000, 'ft_dev': 2000, 'wmt_test': 3003}
idiom sample: {'src': 'Not a word of all this to anyone!', 'tgt': 'Kein Wort von all dem zu irgendjemandem!'}


## Metrics and Generation Helpers

In [ ]:
# Set Global Hyperparameters for Predictions
max_source_len = 256
max_new_tokens = 128
batch_size = 16

In [ ]:
@torch.no_grad()

# Generate Predictions
def generate_preds(model, tok, sources, max_source_len=max_source_len, max_new_tokens=max_new_tokens, batch_size=batch_size):
    model.eval()
    preds=[]
    for i in range(0, len(sources), batch_size):
        batch = sources[i:i+batch_size]
        inputs = tok(batch, return_tensors="pt", truncation=True, padding=True, max_length=max_source_len).to(device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

# Compute Metrics
def compute_metrics(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"bleu": float(bleu), "chrf": float(chrf)}

In [ ]:
# Metrics Logger

METRICS_PATH = os.path.join(RESULTS_DIR, "metrics.csv")

METRIC_FIELDS = [
    "timestamp",
    "run_name",
    "split",
    "bleu",
    "chrf",
    "model_name",
    "learning_rate",
    "epochs",
    "batch_size",
    "max_source_len",
    "max_new_tokens",
    "freeze_encoder",
    "train_size",
    "dev_size",
    "seed",
    "n_eval",
    "notes",
]

def log_metrics(
    run_name,
    split,
    metrics,
    *,
    model_name=None,
    learning_rate=None,
    epochs=None,
    batch_size=None,
    max_source_len=None,
    max_new_tokens=None,
    freeze_encoder=None,
    train_size=None,
    dev_size=None,
    seed=None,
    n_eval=None,
    notes=None,
):
    # Initialize full schema with None
    row = {k: None for k in METRIC_FIELDS}

    row.update({
        "timestamp": datetime.now(ZoneInfo('America/Los_Angeles')).isoformat(timespec="seconds"),
        "run_name": run_name,
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "model_name": model_name,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "batch_size": batch_size,
        "max_source_len": max_source_len,
        "max_new_tokens": max_new_tokens,
        "freeze_encoder": freeze_encoder,
        "train_size": train_size,
        "dev_size": dev_size,
        "seed": seed,
        "n_eval": n_eval,
        "notes": notes,
    })

    df = pd.DataFrame([row])

    if os.path.exists(METRICS_PATH):
        df.to_csv(METRICS_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(METRICS_PATH, index=False)

    return row

# Stage 0: Baseline Evaluation
- Model: Helsinki-NLP/opus-mt-en-de

In [ ]:
# Load Model and Tokenizer
BASE_MODEL = "Helsinki-NLP/opus-mt-en-de"
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
#model.config

In [ ]:
# Set test data
idiom_test = idiom_ds["test"]
wmt_test = general_ds["wmt_test"]

# Global Prediction Hyperparameters
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128
BATCH_SIZE = 16

### Baseline Evaluation

In [ ]:
# Evaluate Idiom Translation
idiom_pred = generate_preds(model, tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=BATCH_SIZE)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("baseline idioms:", idiom_m)

log_metrics(
    "baseline", "idioms_test", idiom_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(idiom_ds["train"]),
    dev_size=len(idiom_ds["dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="pretrained baseline",
)

# Evaluate General Translation
wmt_pred = generate_preds(model, tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=BATCH_SIZE)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("baseline wmt:", wmt_m)

log_metrics(
    "baseline", "wmt_test", wmt_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="pretrained baseline (WMT test full 3003)",
)


baseline idioms: {'bleu': 39.64935194574556, 'chrf': 60.786111999956105}
baseline wmt: {'bleu': 27.567994028871393, 'chrf': 58.43184557779851}


{'timestamp': '2026-02-25T16:53:54-08:00',
 'run_name': 'baseline',
 'split': 'wmt_test',
 'bleu': 27.567994028871393,
 'chrf': 58.43184557779851,
 'model_name': 'Helsinki-NLP/opus-mt-en-de',
 'learning_rate': None,
 'epochs': None,
 'batch_size': 16,
 'max_source_len': 256,
 'max_new_tokens': 128,
 'freeze_encoder': False,
 'train_size': 20000,
 'dev_size': 2000,
 'seed': 42,
 'n_eval': 3003,
 'notes': 'pretrained baseline (WMT test full 3003)'}

## Stage 1: Idiom fine-tuning
This training function ingests a mix of data (90/10) and leaves the top N layers trainable

In [ ]:
# 90/10 mixer for Stage-2 training data
def make_mixed_train_dataset(wmt_train, idiom_train, idiom_ratio=0.10, seed=42):
    """
    Returns a shuffled dataset with:
      (1 - idiom_ratio) * wmt_train  +  idiom_ratio * idiom_train
    Sampling is done WITHOUT replacement from each dataset.
    """
    assert 0.0 <= idiom_ratio <= 1.0

    n_total = len(wmt_train)
    n_idiom = int(n_total * idiom_ratio)
    n_wmt = n_total - n_idiom

    wmt_sample = wmt_train.shuffle(seed=seed).select(range(n_wmt))
    idiom_sample = idiom_train.shuffle(seed=seed).select(range(min(n_idiom, len(idiom_train))))

    mixed = concatenate_datasets([wmt_sample, idiom_sample]).shuffle(seed=seed)
    return mixed

In [ ]:
def freeze_decoder_except_top_n(model, n_trainable=2):
    """
    Freezes ALL decoder layers, then unfreezes only the top n_trainable layers.
    Works for MarianMT-like models with model.model.decoder.layers.
    """
    if not (hasattr(model, "model") and hasattr(model.model, "decoder")):
        print("No decoder found at model.model.decoder; skipping decoder freeze.")
        return

    decoder = model.model.decoder

    # Freeze everything in decoder first
    for p in decoder.parameters():
        p.requires_grad = False

    # Unfreeze top N layers if structure exists
    if hasattr(decoder, "layers"):
        layers = decoder.layers
        n_trainable = max(0, min(n_trainable, len(layers)))
        for layer in layers[-n_trainable:]:
            for p in layer.parameters():
                p.requires_grad = True
        print(f"Decoder: unfroze top {n_trainable}/{len(layers)} layers.")
    else:
        print("Decoder has no .layers attribute; leaving decoder fully frozen.")

    # Usually you want the final projection / lm_head trainable
    if hasattr(model, "lm_head"):
        for p in model.lm_head.parameters():
            p.requires_grad = True

In [ ]:
def fine_tune(
    model_name_or_path,
    train_ds,
    dev_ds,
    out_dir,
    lr,
    epochs,
    batch_size=8,
    freeze_encoder=False,
    decoder_top_layers_trainable=None,
    weight_decay=0.0,
    early_stopping_patience=None,
):
    tok = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path).to(device)

    # ----- Freezing: encoder -----
    if freeze_encoder and hasattr(model, "model") and hasattr(model.model, "encoder"):
        for p in model.model.encoder.parameters():
            p.requires_grad = False
        if hasattr(model.model, "shared"):
            for p in model.model.shared.parameters():
                p.requires_grad = False

    # ----- Freezing: decoder (partial) -----
    if decoder_top_layers_trainable is not None:
        freeze_decoder_except_top_n(model, n_trainable=decoder_top_layers_trainable)

    def tokenize(batch):
        x = tok(batch["src"], truncation=True, max_length=256)
        y = tok(text_target=batch["tgt"], truncation=True, max_length=256)
        x["labels"] = y["input_ids"]
        return x

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
    dev_tok   = dev_ds.map(tokenize, batched=True, remove_columns=dev_ds.column_names)

    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,

        # Evaluation / saving
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # Regularization
        weight_decay=weight_decay,

        # Logging
        logging_steps=50,
        report_to=[],

        fp16=torch.cuda.is_available(),
    )

    collator = DataCollatorForSeq2Seq(tok, model=model)

    callbacks = []
    if early_stopping_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=collator,
        callbacks=callbacks,
    )

    trainer.train()
    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir

## Run fine-tuning

In [ ]:
IDIOM_ONLY_DIR = os.path.join(CHECKPOINT_DIR, "idiom_only_v1")

# Set Fine-tuning Hyperparameters
lr = 5e-5
epochs = 3
batch_size = 8

# Run Fine-tuning
fine_tune(BASE_MODEL, idiom_ds["train"], idiom_ds["dev"], IDIOM_ONLY_DIR, lr=lr, epochs=epochs, batch_size=batch_size)

idiom_model = AutoModelForSeq2SeqLM.from_pretrained(IDIOM_ONLY_DIR).to(device)
idiom_tok = AutoTokenizer.from_pretrained(IDIOM_ONLY_DIR, use_fast=False)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,1.186557,1.116577
2,0.733086,1.105641
3,0.514618,1.111506


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### Stage 1 Evaluation

In [ ]:
# Evaluate Idiom Translation after stage 1 fine-tuning
idiom_pred = generate_preds(idiom_model, idiom_tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=16)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("idiom only:", idiom_m)

log_metrics(
    "idiom_only_v1",
    "idioms_test",
    idiom_m,
    model_name=IDIOM_ONLY_DIR,
    learning_rate=lr,
    epochs=epochs,
    batch_size=batch_size,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(idiom_ds["train"]),
    dev_size=len(idiom_ds["dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="stage1 idiom-only fine-tune",
)

# Evaluate General Translation after stage 1 fine-tuning
wmt_pred = generate_preds(idiom_model, idiom_tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=16)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("idiom_only wmt:", wmt_m)

log_metrics(
    "idiom_only_v1",
    "wmt_test",
    wmt_m,
    model_name=IDIOM_ONLY_DIR,
    learning_rate=lr,
    epochs=epochs,
    batch_size=batch_size,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="stage1 idiom-only fine-tune (eval on WMT)",
)

two_stage_mixed idioms: {'bleu': 44.00220420136346, 'chrf': 64.14987505987591}
idiom_only wmt: {'bleu': 26.83530292293189, 'chrf': 58.30171517973468}


{'timestamp': '2026-02-25T17:28:57-08:00',
 'run_name': 'idiom_only_v1',
 'split': 'wmt_test',
 'bleu': 26.83530292293189,
 'chrf': 58.30171517973468,
 'model_name': '/content/drive/MyDrive/ds266_idiom_mt/checkpoints/idiom_only_v1',
 'learning_rate': 5e-05,
 'epochs': 3,
 'batch_size': 8,
 'max_source_len': 256,
 'max_new_tokens': 128,
 'freeze_encoder': False,
 'train_size': 20000,
 'dev_size': 2000,
 'seed': 42,
 'n_eval': 3003,
 'notes': 'stage1 idiom-only fine-tune (eval on WMT)'}

## Stage 2: General Language Fine-tuning
with Frozen Encoder option and mixed data (90/10)

In [ ]:
STAGE1 = os.path.join(CHECKPOINT_DIR, "two_stage_frozen_v1_stage1")
# STAGE2_MIXED = os.path.join(CHECKPOINT_DIR, "two_stage_frozen_mix_v1_lr2e5_dec4")
STAGE2_WMT_ONLY_FREEZE = os.path.join(CHECKPOINT_DIR,"two_stage_wmt_only_freeze_dec1_lr1e5_v1_stage2")

# Create mixed Stage-2 training dataset
mixed_train = make_mixed_train_dataset(
    wmt_train=general_ds["ft_train"],
    idiom_train=idiom_ds["train"],
    idiom_ratio=0.10,
    seed=SEED
)

In [ ]:
# Training hyperparameters
lr_stage2 = 1e-5
epochs_stage2 = 6
batch_size_stage2 = 8

fine_tune(
    STAGE1,
    general_ds["ft_train"],
    general_ds["ft_dev"],
    STAGE2_WMT_ONLY_FREEZE,
    lr=lr_stage2,
    epochs=epochs_stage2,
    batch_size=batch_size_stage2,
    freeze_encoder=True,
    decoder_top_layers_trainable=1,
    weight_decay=0.0,
    early_stopping_patience=2,
)

wmt_only_model = AutoModelForSeq2SeqLM.from_pretrained(STAGE2_WMT_ONLY_FREEZE).to(device)
wmt_only_tok   = AutoTokenizer.from_pretrained(STAGE2_WMT_ONLY_FREEZE, use_fast=False)



Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Decoder: unfroze top 1/24 layers.


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,2.084428,1.875155
2,1.978867,1.841047
3,1.750779,1.824367
4,1.946248,1.815808
5,1.871700,1.811185
6,1.998401,1.809792


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### Stage 2 Evaluation

In [ ]:
# Idioms test
idiom_pred = generate_preds(wmt_only_model, wmt_only_tok, idiom_test["src"], batch_size=16)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("Stage2 WMT-only idioms:", idiom_m)

log_metrics(
    "two_stage_wmt_only_freeze_dec1_lr1e5_v1",
    "idioms_test",
    idiom_m,
    model_name=STAGE2_MIXED,
    learning_rate=lr_stage2,
    epochs=epochs_stage2,
    batch_size=batch_size_stage2,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=True,
    train_size=len(mixed_train),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes = "stage2: WMT-only | enc_frozen | dec_top=1 | lr=1e-5 | no_wd | ES=2",
)

# WMT test
wmt_pred = generate_preds(wmt_only_model, wmt_only_tok, wmt_test["src"], batch_size=16)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("Stage2 WMT-only wmt:", wmt_m)

log_metrics(
    "two_stage_wmt_only_freeze_dec1_lr1e5_v1",
    "wmt_test",
    wmt_m,
    model_name=STAGE2_MIXED,
    learning_rate=lr_stage2,
    epochs=epochs_stage2,
    batch_size=batch_size_stage2,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=True,
    train_size=len(mixed_train),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes = "stage2: WMT-only | enc_frozen | dec_top=1 | lr=1e-5 | no_wd | ES=2",
)

Stage2 WMT-only idioms: {'bleu': 30.081817822741467, 'chrf': 51.76201709638117}
Stage2 WMT-only wmt: {'bleu': 22.304615505342365, 'chrf': 53.54493896028909}


{'timestamp': '2026-02-25T20:14:54-08:00',
 'run_name': 'two_stage_wmt_only_freeze_dec1_lr1e5_v1',
 'split': 'wmt_test',
 'bleu': 22.304615505342365,
 'chrf': 53.54493896028909,
 'model_name': '/content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_mix_v1_lr2e5_dec4',
 'learning_rate': 1e-05,
 'epochs': 6,
 'batch_size': 8,
 'max_source_len': 256,
 'max_new_tokens': 128,
 'freeze_encoder': True,
 'train_size': 18800,
 'dev_size': 2000,
 'seed': 42,
 'n_eval': 3003,
 'notes': 'stage2: WMT-only | enc_frozen | dec_top=1 | lr=1e-5 | no_wd | ES=2'}